# K-NN Sensitivity Analysis: Type Classifier Quality

## Complete Implementation

**Theoretical Foundation:**
- As K → ∞, k-NN provably converges to Bayes posterior
- We vary K from 1 (extreme overfitting) to 2000 (approaching Bayes)
- Show: Better posterior estimate → Lower queueing cost

**Method:**
1. Use Dense(512) embeddings from trained MobileNet
2. Run k-NN with varying K on embeddings
3. Measure deviation: Cross-entropy vs true labels
4. Evaluate Type-First cost
5. Plot: Deviation vs Cost (monotonic decrease expected)

**Author:** Simrita Singh  
**Date:** January 2026

## Setup

In [5]:
# Data directory - adjust path as needed
DATA_DIR = '../03_chest_xray_case_study/model_weights/'

print(f"DATA_DIR = '{DATA_DIR}'")


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import softmax
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports successful")

✓ Imports successful


## Configuration

In [7]:
# System parameters
T = 14  # Number of types
N = 4   # Number of queues
RHO = 0.9  # System utilization

# Arrival rates (from paper Table 1)
LAMBDA = np.array([0.05, 0.02, 0.01, 0.02, 0.04, 0.09, 0.12, 0.04,
                   0.01, 0.01, 0.01, 0.02, 0.02, 0.54])

# Service rates (identical for all types)
MU = np.sum(LAMBDA) / RHO
MU_VEC = np.ones(T) * MU

# Disease rank order
DISEASE_RANK_ORDER = [
    'Pneumothorax', 'Emphysema', 'Pneumonia', 'Edema',
    'Consolidation', 'Effusion', 'Infiltration', 'Atelectasis',
    'Cardiomegaly', 'Pleural_Thickening', 'Fibrosis', 'Mass',
    'Nodule', 'No Finding'
]

# Delay costs - convex structure c_i = 1.8^(T-i)
DELAY_COSTS = np.array([1.8**(T - i) for i in range(1, T+1)])

print(f"Configuration:")
print(f"  Types: {T}, Queues: {N}")
print(f"  Utilization: {RHO}")
print(f"  Service rate μ: {MU:.3f}")
print(f"  Delay costs: [{DELAY_COSTS[0]:.1f}, ..., {DELAY_COSTS[-1]:.1f}]")

Configuration:
  Types: 14, Queues: 4
  Utilization: 0.9
  Service rate μ: 1.111
  Delay costs: [2082.3, ..., 1.0]


## Load Data

In [9]:
# Data directory - adjust path as needed
DATA_DIR = '../03_chest_xray_case_study/model_weights/'

print(f"DATA_DIR = '{DATA_DIR}'")


In [10]:
print("Loading data...")

# Load Dense(512) features (embeddings)
features = np.load(os.path.join(DATA_DIR, 'dense512_features.npy'))
print(f"  ✓ Features: {features.shape}")

# Load metadata
metadata = pd.read_csv(os.path.join(DATA_DIR, 'image_metadata.csv'))
print(f"  ✓ Metadata: {metadata.shape}")

# Extract true type labels
def extract_true_types(metadata):
    """Extract true type (highest-ranked disease present)."""
    n_images = len(metadata)
    true_types = np.zeros(n_images, dtype=int)

    for idx, row in metadata.iterrows():
        finding_labels = str(row['Finding Labels'])
        has_disease = [disease in finding_labels for disease in DISEASE_RANK_ORDER]

        for type_idx in range(T):
            if has_disease[type_idx]:
                true_types[idx] = type_idx
                break

    return true_types

true_types = extract_true_types(metadata)
print(f"  ✓ True types: {true_types.shape}")

# Type distribution
print(f"\nType distribution:")
for i in [0, 1, 2, 12, 13]:  # Show key types
    count = np.sum(true_types == i)
    pct = 100 * count / len(true_types)
    print(f"  Type {i+1:2d} ({DISEASE_RANK_ORDER[i]:20s}): {count:6d} ({pct:5.2f}%)")

Loading data...
  ✓ Features: (102120, 512)
  ✓ Metadata: (102120, 6)
  ✓ True types: (102120,)

Type distribution:
  Type  1 (Pneumothorax        ):   4950 ( 4.85%)
  Type  2 (Emphysema           ):   1602 ( 1.57%)
  Type  3 (Pneumonia           ):   1243 ( 1.22%)
  Type 13 (Nodule              ):   2501 ( 2.45%)
  Type 14 (No Finding          ):  54652 (53.52%)


## Train/Test Split

In [11]:
print("Splitting train/test (80/20, stratified by type)...")

indices = np.arange(len(features))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=true_types,
    random_state=42
)

features_train = features[train_idx]
features_test = features[test_idx]
types_train = true_types[train_idx]
types_test = true_types[test_idx]

print(f"  Train: {len(train_idx):6d} images")
print(f"  Test:  {len(test_idx):6d} images")

Splitting train/test (80/20, stratified by type)...
  Train:  81696 images
  Test:   20424 images


## Core Queueing Functions (Fixed Priority Order)

In [12]:
def compute_mean_residual(lambda_vec, mu_vec):
    """E[R] = Σ λ_i / μ_i²"""
    return np.sum(lambda_vec / (mu_vec ** 2))

def compute_queue_utilization(j, P_matrix, lambda_vec, mu_vec):
    """ρ(j) = Σ_i λ_i * P_ij / μ_i"""
    return np.sum(lambda_vec * P_matrix[:, j] / mu_vec)

def compute_waiting_time_queue_ik(i, j, P_matrix, lambda_vec, mu_vec):
    """
    Waiting time contribution for type i in queue j.
    Uses FIXED priority: Queue 0 > Queue 1 > Queue 2 > Queue 3
    """
    pos = j
    E_R = compute_mean_residual(lambda_vec, mu_vec)

    if pos == 0:
        den1 = 1.0
        rho_cumsum = compute_queue_utilization(0, P_matrix, lambda_vec, mu_vec)
        den2 = 1.0 - rho_cumsum
    else:
        rho_before = sum([compute_queue_utilization(l, P_matrix, lambda_vec, mu_vec)
                         for l in range(pos)])
        den1 = 1.0 - rho_before

        rho_upto = sum([compute_queue_utilization(l, P_matrix, lambda_vec, mu_vec)
                       for l in range(pos + 1)])
        den2 = 1.0 - rho_upto

    if den1 <= 0 or den2 <= 0:
        return 1000.0 * lambda_vec[i] * P_matrix[i, j]

    return lambda_vec[i] * P_matrix[i, j] * E_R / (den1 * den2)

def compute_cost(P_matrix, lambda_vec, mu_vec, delay_costs):
    """Total average waiting cost with FIXED priority [0, 1, 2, 3]."""
    T, N = P_matrix.shape

    total_cost = 0.0
    for i in range(T):
        for j in range(N):
            wait_ij = compute_waiting_time_queue_ik(i, j, P_matrix, lambda_vec, mu_vec)
            total_cost += delay_costs[i] * wait_ij

    return total_cost

print("✓ Core queueing functions loaded (FIXED priority)")

✓ Core queueing functions loaded (FIXED priority)


## Type-First Functions

In [13]:
def type_to_queue_probs(type_probs, beta):
    """Map type probabilities to queue probabilities using softmax."""
    logits = np.dot(type_probs, beta.T)
    logits_with_baseline = np.concatenate([logits, np.zeros((len(type_probs), 1))], axis=1)
    return softmax(logits_with_baseline, axis=1)

def compute_empirical_P_hat(type_probs, beta):
    """
    Compute EMPIRICAL classification matrix P^t_hat for OPTIMIZATION.
    Weighted by PREDICTED type probabilities.
    """
    queue_probs = type_to_queue_probs(type_probs, beta)
    T = type_probs.shape[1]
    N = queue_probs.shape[1]

    P_hat = np.zeros((T, N))
    for i in range(T):
        numerator = np.sum(type_probs[:, i:i+1] * queue_probs, axis=0)
        denominator = np.sum(type_probs[:, i])

        if denominator > 0:
            P_hat[i, :] = numerator / denominator
        else:
            P_hat[i, :] = 1.0 / N

    return P_hat

def compute_true_P(type_probs, beta, true_types):
    """
    Compute TRUE classification matrix P for EVALUATION.
    Grouped by TRUE types.
    """
    queue_probs = type_to_queue_probs(type_probs, beta)
    T = int(np.max(true_types)) + 1
    N = queue_probs.shape[1]

    P_true = np.zeros((T, N))
    for i in range(T):
        mask = (true_types == i)
        if np.sum(mask) > 0:
            P_true[i, :] = np.mean(queue_probs[mask, :], axis=0)
        else:
            P_true[i, :] = 1.0 / N

    return P_true

def optimize_beta(type_probs, lambda_vec, mu_vec, delay_costs, init_beta=None, verbose=False):
    """Optimize β using EMPIRICAL matrix P^t_hat."""
    T = type_probs.shape[1]

    if init_beta is None:
        init_beta = np.random.randn(3, T) * 0.1

    def objective(beta_flat):
        beta = beta_flat.reshape(3, T)
        P_hat = compute_empirical_P_hat(type_probs, beta)
        return compute_cost(P_hat, lambda_vec, mu_vec, delay_costs)

    result = minimize(objective, init_beta.flatten(), method='L-BFGS-B',
                     options={'maxiter': 100, 'disp': verbose})

    return result.x.reshape(3, T), result.fun

def evaluate_true_cost(type_probs, beta, true_types, lambda_vec, mu_vec, delay_costs):
    """Evaluate TRUE cost using TRUE classification matrix P."""
    P_true = compute_true_P(type_probs, beta, true_types)
    return compute_cost(P_true, lambda_vec, mu_vec, delay_costs)

print("✓ Type-first functions loaded")

✓ Type-first functions loaded


## Deviation Metrics

In [14]:
def compute_cross_entropy(type_probs, true_types):
    """Cross-entropy: -mean(log(P[true_type]))"""
    epsilon = 1e-10
    n = len(true_types)

    ce = 0.0
    for i in range(n):
        true_type = true_types[i]
        prob = type_probs[i, true_type]
        ce += -np.log(prob + epsilon)

    return ce / n

def compute_accuracy(type_probs, true_types):
    """Accuracy: % correct predictions"""
    predicted_types = np.argmax(type_probs, axis=1)
    return np.mean(predicted_types == true_types)

print("✓ Deviation metrics loaded")

✓ Deviation metrics loaded


## Benchmarks

In [15]:
print("="*70)
print("BENCHMARKS (from paper)")
print("="*70)

# Benchmark costs (from your table)
cost_perfect = 158
cost_direct = 395
cost_fifo = 1285

print(f"\nCost structure: c_i = 1.8^(T-i)")
print(f"\nBenchmark costs:")
print(f"  Perfect:  {cost_perfect}")
print(f"  Direct:   {cost_direct}")
print(f"  FIFO:     {cost_fifo}")
print(f"  Range:    {cost_fifo - cost_perfect}")

print("\n✓ Ready for k-NN sensitivity analysis\n")

BENCHMARKS (from paper)

Cost structure: c_i = 1.8^(T-i)

Benchmark costs:
  Perfect:  158
  Direct:   395
  FIFO:     1285
  Range:    1127

✓ Ready for k-NN sensitivity analysis



## K-NN Sensitivity Analysis

**Experiment:**
- Vary K from 1 (extreme overfitting) to 2000 (approaching Bayes)
- For each K: compute posteriors, measure deviation, evaluate cost
- Expected: As K ↑, cross-entropy ↓, cost ↓

In [16]:
print("="*70)
print("K-NN SENSITIVITY ANALYSIS")
print("="*70)
print("\nVarying K to show: Better posterior → Lower cost")
print("Theoretical foundation: k-NN → Bayes posterior as K → ∞")
print("")

# Define K values to test (including extreme K=1)
K_VALUES = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000]

results = []

for K in K_VALUES:
    print(f"\n{'='*70}")
    print(f"K = {K}")
    print(f"{'='*70}")

    # ========================================================================
    # Step 1: Fit k-NN on training embeddings
    # ========================================================================
    print(f"  Fitting k-NN with K={K}...")
    knn = NearestNeighbors(n_neighbors=K, metric='euclidean', n_jobs=-1)
    knn.fit(features_train)

    # ========================================================================
    # Step 2: Predict type posteriors on test set
    # ========================================================================
    print(f"  Finding {K} nearest neighbors for test set...")
    distances, indices = knn.kneighbors(features_test)

    print(f"  Computing empirical posteriors...")
    n_test = len(features_test)
    P_knn = np.zeros((n_test, T))

    for i in range(n_test):
        if i % 5000 == 0 and i > 0:
            print(f"    Progress: {i}/{n_test}")

        # Get types of K nearest neighbors
        neighbor_types = types_train[indices[i]]

        # Empirical posterior: P(type=j) = count(neighbors with type j) / K
        for t in range(T):
            P_knn[i, t] = np.sum(neighbor_types == t) / K

    print(f"  ✓ Posteriors computed")

    # ========================================================================
    # Step 3: Measure deviation (Cross-Entropy)
    # ========================================================================
    cross_entropy = compute_cross_entropy(P_knn, types_test)
    accuracy = compute_accuracy(P_knn, types_test)

    print(f"  Deviation metrics:")
    print(f"    Cross-entropy: {cross_entropy:.4f}")
    print(f"    Accuracy: {100*accuracy:.2f}%")

    # ========================================================================
    # Step 4: Optimize β using empirical P_hat
    # ========================================================================
    print(f"  Optimizing β...")
    beta_opt, estimated_cost = optimize_beta(P_knn, LAMBDA, MU_VEC, DELAY_COSTS)

    # ========================================================================
    # Step 5: Evaluate true cost using P_true
    # ========================================================================
    P_true = compute_true_P(P_knn, beta_opt, types_test)
    true_cost = compute_cost(P_true, LAMBDA, MU_VEC, DELAY_COSTS)

    print(f"  Type-First cost:")
    print(f"    Estimated (P_hat): {estimated_cost:.2f}")
    print(f"    True (P_true): {true_cost:.2f}")
    print(f"    Gap: {true_cost - estimated_cost:.2f}")

    # Gap from benchmarks
    gap_from_perfect = true_cost - cost_perfect
    gap_from_direct = true_cost - cost_direct
    print(f"  Gaps from benchmarks:")
    print(f"    vs Perfect: +{gap_from_perfect:.0f}")
    print(f"    vs Direct: {gap_from_direct:+.0f}")

    # ========================================================================
    # Record results
    # ========================================================================
    results.append({
        'K': K,
        'cross_entropy': cross_entropy,
        'accuracy': accuracy,
        'estimated_cost': estimated_cost,
        'true_cost': true_cost,
        'gap': true_cost - estimated_cost,
        'gap_from_perfect': gap_from_perfect,
        'gap_from_direct': gap_from_direct
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)

print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)
print(results_df[['K', 'accuracy', 'cross_entropy', 'true_cost']].to_string(index=False))

# Check monotonicity
print("\n" + "="*70)
print("QUALITY CHECKS")
print("="*70)

is_monotonic = all(results_df['true_cost'].iloc[i] >= results_df['true_cost'].iloc[i+1]
                  for i in range(len(results_df)-1))
print(f"\nMonotonic decrease in cost: {is_monotonic}")

# Correlation
corr = results_df[['cross_entropy', 'true_cost']].corr().iloc[0, 1]
print(f"Correlation (CE vs Cost): {corr:.4f}")

# Range
cost_range = results_df['true_cost'].max() - results_df['true_cost'].min()
print(f"\nCost range: {cost_range:.0f} ({results_df['true_cost'].max():.0f} → {results_df['true_cost'].min():.0f})")
print(f"  K=1 (worst):  {results_df.iloc[0]['true_cost']:.0f}")
print(f"  K=2000 (best): {results_df.iloc[-1]['true_cost']:.0f}")

# Save results
results_df.to_csv('knn_sensitivity_results.csv', index=False)
print("\n✓ Saved: knn_sensitivity_results.csv")

# Save to Google Drive if using it
if DATA_DIR != '.':
    drive_path = os.path.join(DATA_DIR, 'knn_sensitivity_results.csv')
    results_df.to_csv(drive_path, index=False)
    print(f"✓ Saved to Drive: {drive_path}")

K-NN SENSITIVITY ANALYSIS

Varying K to show: Better posterior → Lower cost
Theoretical foundation: k-NN → Bayes posterior as K → ∞


K = 1
  Fitting k-NN with K=1...
  Finding 1 nearest neighbors for test set...
  Computing empirical posteriors...
    Progress: 5000/20424
    Progress: 10000/20424
    Progress: 15000/20424
    Progress: 20000/20424
  ✓ Posteriors computed
  Deviation metrics:
    Cross-entropy: 13.6065
    Accuracy: 40.91%
  Optimizing β...
  Type-First cost:
    Estimated (P_hat): 167.34
    True (P_true): 991.89
    Gap: 824.55
  Gaps from benchmarks:
    vs Perfect: +834
    vs Direct: +597

K = 2
  Fitting k-NN with K=2...
  Finding 2 nearest neighbors for test set...
  Computing empirical posteriors...
    Progress: 5000/20424
    Progress: 10000/20424
    Progress: 15000/20424
    Progress: 20000/20424
  ✓ Posteriors computed
  Deviation metrics:
    Cross-entropy: 10.2884
    Accuracy: 32.95%
  Optimizing β...
  Type-First cost:
    Estimated (P_hat): 184.63
  

In [25]:
# ============================================================================
# INSPECT HDF5 FILE STRUCTURE
# ============================================================================

import h5py

weights_path = os.path.join(DATA_DIR, 'xray_class_weights.best.direct_gp_r_1.8_rho_0.90.december_2023.hdf5')

print("="*70)
print("INSPECTING HDF5 FILE STRUCTURE")
print("="*70)

def print_structure(name, obj):
    """Recursively print HDF5 structure."""
    if isinstance(obj, h5py.Dataset):
        print(f"  Dataset: {name:50s} shape={obj.shape}")
    elif isinstance(obj, h5py.Group):
        print(f"  Group:   {name}")

with h5py.File(weights_path, 'r') as f:
    print("\nFull structure:")
    f.visititems(print_structure)

    print("\n" + "="*70)
    print("DENSE LAYERS IN DETAIL")
    print("="*70)

    # Look at all dense layers
    for layer_name in f.keys():
        if 'dense' in layer_name.lower():
            print(f"\n{layer_name}:")
            layer = f[layer_name][layer_name]

            if 'kernel:0' in layer:
                kernel = layer['kernel:0'][:]
                bias = layer['bias:0'][:]
                print(f"  kernel:0 shape: {kernel.shape}")
                print(f"  bias:0 shape: {bias.shape}")
                print(f"  kernel sample: {kernel[:3, :2]}")  # Show first few values
            else:
                print(f"  Available keys: {list(layer.keys())}")

    print("\n" + "="*70)
    print("ALL TOP-LEVEL KEYS")
    print("="*70)
    for key in f.keys():
        print(f"  {key}")

INSPECTING HDF5 FILE STRUCTURE

Full structure:
  Group:   dense
  Group:   dense/dense
  Dataset: dense/dense/bias:0                                 shape=(512,)
  Dataset: dense/dense/kernel:0                               shape=(1024, 512)
  Group:   dense_1
  Group:   dense_1/dense_1
  Dataset: dense_1/dense_1/bias:0                             shape=(14,)
  Dataset: dense_1/dense_1/kernel:0                           shape=(512, 14)
  Group:   dense_2
  Group:   dense_2/dense_2
  Dataset: dense_2/dense_2/bias:0                             shape=(4,)
  Dataset: dense_2/dense_2/kernel:0                           shape=(14, 4)
  Group:   dropout
  Group:   dropout_1
  Group:   global_average_pooling2d
  Group:   mobilenet_1.00_512
  Group:   mobilenet_1.00_512/conv1
  Dataset: mobilenet_1.00_512/conv1/kernel:0                  shape=(3, 3, 3, 32)
  Group:   mobilenet_1.00_512/conv1_bn
  Dataset: mobilenet_1.00_512/conv1_bn/beta:0                 shape=(32,)
  Dataset: mobilenet_1.00_5

## Visualization

In [ ]:
# ============================================================================
# FIGURE: K-NN SENSITIVITY (2 plots)
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ============================================================================
# Plot 1: Cross-Entropy vs Cost (Main Result)
# ============================================================================
ax1 = axes[0]

ax1.plot(results_df['cross_entropy'], results_df['true_cost'],
        marker='o', markersize=10, linewidth=3, color='steelblue',
        label='Type-First C(P^t)', zorder=3)

# Add K labels to selected points
label_points = [0, 2, 5, 8, 10]  # Indices for K=1, 5, 100, 1000, 2000
for idx in label_points:
    if idx < len(results_df):
        row = results_df.iloc[idx]
        ax1.annotate(f"K={row['K']}",
                    xy=(row['cross_entropy'], row['true_cost']),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=10, alpha=0.8, fontweight='bold')

# Benchmarks
ax1.axhline(y=cost_perfect, color='green', linestyle='--', linewidth=2.5, alpha=0.7,
           label=f'Perfect = {cost_perfect}', zorder=1)
ax1.axhline(y=cost_direct, color='purple', linestyle='--', linewidth=2.5, alpha=0.7,
           label=f'Direct = {cost_direct}', zorder=1)
ax1.axhline(y=cost_fifo, color='red', linestyle='--', linewidth=2.5, alpha=0.7,
           label=f'FIFO = {cost_fifo}', zorder=1)

# Labels and formatting
ax1.set_xlabel('Cross-Entropy (Deviation from True Posterior)',
              fontsize=14, fontweight='bold')
ax1.set_ylabel('Average Waiting Cost', fontsize=14, fontweight='bold')
ax1.set_title('Type-First Cost vs Posterior Quality\n(k-NN with varying K)',
             fontsize=15, fontweight='bold')
ax1.legend(fontsize=11, loc='upper right')
ax1.grid(True, alpha=0.3)

# ============================================================================
# Plot 2: K vs Cost (Convergence)
# ============================================================================
ax2 = axes[1]

ax2.semilogx(results_df['K'], results_df['true_cost'],
            marker='o', markersize=10, linewidth=3, color='steelblue',
            label='Type-First C(P^t)', zorder=3)

# Benchmarks
ax2.axhline(y=cost_perfect, color='green', linestyle='--', linewidth=2.5, alpha=0.7,
           label=f'Perfect = {cost_perfect}', zorder=1)
ax2.axhline(y=cost_direct, color='purple', linestyle='--', linewidth=2.5, alpha=0.7,
           label=f'Direct = {cost_direct}', zorder=1)
ax2.axhline(y=cost_fifo, color='red', linestyle='--', linewidth=2.5, alpha=0.7,
           label=f'FIFO = {cost_fifo}', zorder=1)

# Annotations
ax2.annotate('K=1\n(worst)',
           xy=(results_df.iloc[0]['K'], results_df.iloc[0]['true_cost']),
           xytext=(1.5, results_df.iloc[0]['true_cost'] + 100),
           fontsize=11, ha='left',
           arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax2.annotate('Large K\n→ Bayes',
           xy=(results_df.iloc[-1]['K'], results_df.iloc[-1]['true_cost']),
           xytext=(1000, results_df.iloc[-1]['true_cost'] - 150),
           fontsize=11, ha='center',
           arrowprops=dict(arrowstyle='->', color='green', lw=1.5))

# Labels and formatting
ax2.set_xlabel('K (Number of Neighbors)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Average Waiting Cost', fontsize=14, fontweight='bold')
ax2.set_title('Convergence: As K → ∞, k-NN → Bayes Posterior',
             fontsize=15, fontweight='bold')
ax2.legend(fontsize=11, loc='upper right')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('knn_sensitivity_analysis.png', dpi=300, bbox_inches='tight')
plt.savefig('knn_sensitivity_analysis.pdf', bbox_inches='tight')
print("✓ Saved: knn_sensitivity_analysis.png/pdf")

# Save to Google Drive if using it
if DATA_DIR != '.':
    plt.savefig(os.path.join(DATA_DIR, 'knn_sensitivity_analysis.png'), dpi=300, bbox_inches='tight')
    print(f"✓ Saved to Drive")

plt.show()

## Additional Analysis: Accuracy vs Cost

In [ ]:
# Single plot: Accuracy vs Cost (most interpretable)
fig, ax = plt.subplots(1, 1, figsize=(10, 7))

ax.plot(100*results_df['accuracy'], results_df['true_cost'],
       marker='o', markersize=10, linewidth=3, color='steelblue',
       label='Type-First C(P^t)')

# Add K labels
for idx in [0, 2, 5, 8, 10]:
    if idx < len(results_df):
        row = results_df.iloc[idx]
        ax.annotate(f"K={row['K']}",
                   xy=(100*row['accuracy'], row['true_cost']),
                   xytext=(5, 5), textcoords='offset points',
                   fontsize=10, alpha=0.8, fontweight='bold')

# Benchmarks
ax.axhline(y=cost_perfect, color='green', linestyle='--', linewidth=2.5, alpha=0.7,
          label=f'Perfect = {cost_perfect}')
ax.axhline(y=cost_direct, color='purple', linestyle='--', linewidth=2.5, alpha=0.7,
          label=f'Direct = {cost_direct}')
ax.axhline(y=cost_fifo, color='red', linestyle='--', linewidth=2.5, alpha=0.7,
          label=f'FIFO = {cost_fifo}')

ax.set_xlabel('Type Classifier Accuracy (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Waiting Cost', fontsize=14, fontweight='bold')
ax.set_title('Type-First Performance vs Classifier Accuracy\n(k-NN on Dense512 Embeddings)',
            fontsize=15, fontweight='bold')
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_vs_cost.png', dpi=300, bbox_inches='tight')
plt.savefig('accuracy_vs_cost.pdf', bbox_inches='tight')
print("✓ Saved: accuracy_vs_cost.png/pdf")
plt.show()

## Summary Statistics

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

print(f"\n📊 Benchmarks:")
print(f"  Perfect:  {cost_perfect}")
print(f"  Direct:   {cost_direct}")
print(f"  FIFO:     {cost_fifo}")

print(f"\n📈 K-NN Results (Cost Structure: c_i = 1.8^(T-i)):")
print(f"\n  K=1 (extreme overfitting):")
print(f"    Accuracy: {100*results_df.iloc[0]['accuracy']:.1f}%")
print(f"    Cost: {results_df.iloc[0]['true_cost']:.0f}")

best_idx = results_df['true_cost'].idxmin()
print(f"\n  K={results_df.iloc[best_idx]['K']} (best):")
print(f"    Accuracy: {100*results_df.iloc[best_idx]['accuracy']:.1f}%")
print(f"    Cost: {results_df.iloc[best_idx]['true_cost']:.0f}")

print(f"\n  Improvement:")
improvement = results_df.iloc[0]['true_cost'] - results_df.iloc[best_idx]['true_cost']
print(f"    Cost reduction: {improvement:.0f} ({100*improvement/results_df.iloc[0]['true_cost']:.1f}%)")

gap_closed = (results_df.iloc[0]['true_cost'] - results_df.iloc[best_idx]['true_cost']) / (cost_fifo - cost_perfect)
print(f"    Gap closed (K=1 → K={results_df.iloc[best_idx]['K']}): {100*gap_closed:.1f}%")

print(f"\n🎯 Key Finding:")
print(f"  Strong negative correlation (r={corr:.3f}) confirms:")
print(f"  Better type posterior → Lower queueing cost")
print(f"  As K → ∞, k-NN approaches Bayes posterior (proven theoretically)")

print("\n" + "="*70)
print("✓ ANALYSIS COMPLETE")
print("="*70)